In [1]:
from typing import TypedDict 


In [ ]:
class CricketState(TypedDict): 
    # Input fields 
    runs: int 
    balls: int 
    fours: int 
    sixes: int 
     
    # Processed fields (calculated in parallel) 
    strike_rate: float 
    boundary_percentage: float 
    balls_per_boundary: float 
     
    # Aggregated output 
    summary: str 


In [ ]:
def calc_strike_rate(state: CricketState): 
    sr = (state['runs'] / state['balls']) * 100 
    return {"strike_rate": round(sr, 2)} 
 
def calc_boundary_percentage(state: CricketState): 
    boundary_runs = (state['fours'] * 4) + (state['sixes'] * 6) 
    b_pct = (boundary_runs / state['runs']) * 100 
    return {"boundary_percentage": round(b_pct, 2)} 
 
def calc_balls_per_boundary(state: CricketState): 
    total_boundaries = state['fours'] + state['sixes'] 
    bpb = state['balls'] / total_boundaries if total_boundaries > 0 else 0 
    return {"balls_per_boundary": round(bpb, 2)} 
 
def aggregate_summary(state: CricketState): 
    summary_text = ( 
        f"Performance Summary:\n" 
        f"- Strike Rate: {state['strike_rate']}\n" 
        f"- Boundary Runs %: {state['boundary_percentage']}%\n" 
        f"- Balls Per Boundary: {state['balls_per_boundary']}" 
    ) 
    return {"summary": summary_text} 
 

In [ ]:
from langgraph.graph import StateGraph, START, END 
 
# Initialize graph with state schema 
builder = StateGraph(CricketState) 
 
# Add nodes 
builder.add_node("calc_sr", calc_strike_rate) 
builder.add_node("calc_bpct", calc_boundary_percentage) 
builder.add_node("calc_bpb", calc_balls_per_boundary) 
builder.add_node("aggregate", aggregate_summary) 
 
# Fan-Out: Connect START to all three parallel nodes 
builder.add_edge(START, "calc_sr") 
builder.add_edge(START, "calc_bpct") 
builder.add_edge(START, "calc_bpb") 
 
# Fan-In: Connect all parallel nodes to the single aggregation node 
builder.add_edge("calc_sr", "aggregate") 
builder.add_edge("calc_bpct", "aggregate") 
builder.add_edge("calc_bpb", "aggregate") 
 
# Connect aggregation node to END 
builder.add_edge("aggregate", END) 
 
# Compile graph 
graph = builder.compile() 


In [ ]:
graph

In [ ]:
initial_input = { 
    "runs": 100, 
    "balls": 50, 
    "fours": 10, 
    "sixes": 5 
} 
 
result = graph.invoke(initial_input) 
print(result["summary"]) 
 